# 01 Data Discovery

Profile each raw Olist table independently. Record row counts, column names, data types, grain, candidate keys, missing values, duplicates, and relationship notes.

At this stage, we are not cleaning yet. The goal is to understand what each table contains and how the tables relate to each other.

## Setup and Load Raw Tables

In [30]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DATA_DIR = Path("../data/raw")

# Loading the 9 Olist datasets
orders = pd.read_csv(RAW_DATA_DIR / "olist_orders_dataset.csv")
customers = pd.read_csv(RAW_DATA_DIR / "olist_customers_dataset.csv")
geolocation = pd.read_csv(RAW_DATA_DIR / "olist_geolocation_dataset.csv")
order_items = pd.read_csv(RAW_DATA_DIR / "olist_order_items_dataset.csv")
order_payments = pd.read_csv(RAW_DATA_DIR / "olist_order_payments_dataset.csv")
order_reviews = pd.read_csv(RAW_DATA_DIR / "olist_order_reviews_dataset.csv")
products = pd.read_csv(RAW_DATA_DIR / "olist_products_dataset.csv")
sellers = pd.read_csv(RAW_DATA_DIR / "olist_sellers_dataset.csv")
categories = pd.read_csv(RAW_DATA_DIR / "product_category_name_translation.csv")

tables = {
    "orders": orders,
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "products": products,
    "sellers": sellers,
    "categories": categories,
}

In [60]:
tables.get("orders")

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00
...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28 00:00:00
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02 00:00:00
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27 00:00:00
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00


## Reusable Discovery Function

This function runs the same basic checks for every table so the notebook stays consistent.

In [31]:
def discover_table(df, table_name):
    print(f"{table_name} dataset shape: {df.shape}")
    print(f"Number of duplicate rows in {table_name} dataset: {df.duplicated().sum()}")
    print("Column data types:")
    display(df.dtypes.to_frame("dtype"))
    print("Missing values:")
    display(df.isna().sum().to_frame("missing_values"))
    print("Unique values per column:")
    display(df.nunique().sort_values(ascending=False).to_frame("unique_values"))
    print("First 5 rows:")
    display(df.head())
    print("Description:")
    display(df.describe(include="all").T)

## Dataset Inventory

In [32]:
inventory = []

for table_name, df in tables.items():
    inventory.append({
        "table": table_name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicate_rows": int(df.duplicated().sum()),
        "columns_list": ", ".join(df.columns),
    })

inventory_df = pd.DataFrame(inventory)
inventory_df

,table,rows,columns,duplicate_rows,columns_list
0,orders,99441,8,0,"order_id, customer_id, order_status, order_pur..."
1,customers,99441,5,0,"customer_id, customer_unique_id, customer_zip_..."
2,geolocation,1000163,5,261831,"geolocation_zip_code_prefix, geolocation_lat, ..."
3,order_items,112650,7,0,"order_id, order_item_id, product_id, seller_id..."
4,order_payments,103886,5,0,"order_id, payment_sequential, payment_type, pa..."
5,order_reviews,99224,7,0,"review_id, order_id, review_score, review_comm..."
6,products,32951,9,0,"product_id, product_category_name, product_nam..."
7,sellers,3095,4,0,"seller_id, seller_zip_code_prefix, seller_city..."
8,categories,71,2,0,"product_category_name, product_category_name_e..."


## 1. Orders Table

**Grain:** one row per order.

**Candidate key:** `order_id`.

**Relationship notes:** `customer_id` joins to `customers.customer_id`; `order_id` joins to `order_items`, `order_payments`, and `order_reviews`.

The timestamp columns load as `object` from CSV. That is normal during raw discovery; later they should be converted to datetime for EDA and validation.

In [33]:
discover_table(orders, "orders")

orders dataset shape: (99441, 8)
Number of duplicate rows in orders dataset: 0
Column data types:


,dtype
order_id,object
customer_id,object
order_status,object
order_purchase_timestamp,object
order_approved_at,object
order_delivered_carrier_date,object
order_delivered_customer_date,object
order_estimated_delivery_date,object


Missing values:


,missing_values
order_id,0
customer_id,0
order_status,0
order_purchase_timestamp,0
order_approved_at,160
order_delivered_carrier_date,1783
order_delivered_customer_date,2965
order_estimated_delivery_date,0


Unique values per column:


,unique_values
order_id,99441
customer_id,99441
order_purchase_timestamp,98875
order_delivered_customer_date,95664
order_approved_at,90733
order_delivered_carrier_date,81018
order_estimated_delivery_date,459
order_status,8


First 5 rows:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


Description:


,count,unique,top,freq
order_id,99441,99441,e481f51cbdc54678b7cc49136f2d6af7,1
customer_id,99441,99441,9ef432eb6251297304e76186b10a928d,1
order_status,99441,8,delivered,96478
order_purchase_timestamp,99441,98875,2018-04-11 10:48:14,3
order_approved_at,99281,90733,2018-02-27 04:31:10,9
order_delivered_carrier_date,97658,81018,2018-05-09 15:48:00,47
order_delivered_customer_date,96476,95664,2018-05-08 23:38:46,3
order_estimated_delivery_date,99441,459,2017-12-20 00:00:00,522


In [34]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

### Orders Discovery Summary

| Metric | Finding |
|---|---|
| Grain | One row per order |
| Candidate key | `order_id` |
| Important columns | `order_id`, `customer_id`, `order_status`, order timestamp fields |
| Expected relationships | `customer_id` -> `customers.customer_id`; `order_id` -> item/payment/review tables |
| Early quality notes | Delivery and approval dates can be missing depending on order status |
| Next checks | Convert timestamp columns to datetime; inspect date ranges; compare missing delivery dates with `order_status` |

## 2. Customers Table

**Grain:** one row per customer/order-linked customer record.

**Candidate key:** `customer_id`.

**Relationship notes:** joins to `orders.customer_id`. `customer_unique_id` can be used later to analyze repeat customers.

In [35]:
discover_table(customers, "customers")

customers dataset shape: (99441, 5)
Number of duplicate rows in customers dataset: 0
Column data types:


,dtype
customer_id,object
customer_unique_id,object
customer_zip_code_prefix,int64
customer_city,object
customer_state,object


Missing values:


,missing_values
customer_id,0
customer_unique_id,0
customer_zip_code_prefix,0
customer_city,0
customer_state,0


Unique values per column:


,unique_values
customer_id,99441
customer_unique_id,96096
customer_zip_code_prefix,14994
customer_city,4119
customer_state,27


First 5 rows:


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


Description:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customer_id,99441,99441,06b8999e2fba1a1fbc88172c00ba8bc7,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_unique_id,99441,96096,8d50f5eadf50201ccdcedfb9e2ac8455,17,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_zip_code_prefix,99441.0,NaN,NaN,NaN,35137.474583,29797.938996,1003.0,11347.0,24416.0,58900.0,99990.0
customer_city,99441,4119,sao paulo,15540,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_state,99441,27,SP,41746,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [36]:
customers[["customer_state", "customer_city"]].describe(include="all")

,customer_state,customer_city
count,99441,99441
unique,27,4119
top,SP,sao paulo
freq,41746,15540


In [37]:
customers["customer_state"].value_counts().head(10)

customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
Name: count, dtype: int64

### Customers Discovery Summary

| Metric | Finding |
|---|---|
| Grain | One row per customer/order-linked customer record |
| Candidate key | `customer_id` |
| Business key for repeat behavior | `customer_unique_id` |
| Important columns | customer ID fields, zip code prefix, city, state |
| Expected relationships | `customer_id` -> `orders.customer_id`; zip/state/city can support geography analysis |
| Next checks | Compare `customer_id` count with orders; inspect repeat purchases by `customer_unique_id` |

## 3. Geolocation Table

**Grain:** one row per geolocation record for a zip code prefix, city, state, latitude, and longitude combination.

**Candidate key:** no simple unique key expected. Zip code prefixes can appear multiple times.

**Relationship notes:** can enrich customers and sellers through zip code prefixes, but should be handled carefully because repeated prefixes can duplicate rows.

In [38]:
discover_table(geolocation, "geolocation")

geolocation dataset shape: (1000163, 5)
Number of duplicate rows in geolocation dataset: 261831
Column data types:


,dtype
geolocation_zip_code_prefix,int64
geolocation_lat,float64
geolocation_lng,float64
geolocation_city,object
geolocation_state,object


Missing values:


,missing_values
geolocation_zip_code_prefix,0
geolocation_lat,0
geolocation_lng,0
geolocation_city,0
geolocation_state,0


Unique values per column:


,unique_values
geolocation_lng,717613
geolocation_lat,717360
geolocation_zip_code_prefix,19015
geolocation_city,8011
geolocation_state,27


First 5 rows:


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


Description:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
geolocation_zip_code_prefix,1000163.0,NaN,NaN,NaN,36574.166466,30549.33571,1001.0,11075.0,26530.0,63504.0,99990.0
geolocation_lat,1000163.0,NaN,NaN,NaN,-21.176153,5.715866,-36.605374,-23.603546,-22.919377,-19.97962,45.065933
geolocation_lng,1000163.0,NaN,NaN,NaN,-46.390541,4.269748,-101.466766,-48.573172,-46.637879,-43.767709,121.105394
geolocation_city,1000163,8011,sao paulo,135800,NaN,NaN,NaN,NaN,NaN,NaN,NaN
geolocation_state,1000163,27,SP,404268,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [39]:
geolocation["geolocation_state"].value_counts().head(10)

geolocation_state
SP    404268
MG    126336
RJ    121169
RS     61851
PR     57859
SC     38328
BA     36045
GO     20139
ES     16748
PE     16432
Name: count, dtype: int64

In [40]:
geolocation["geolocation_zip_code_prefix"].duplicated().sum()

np.int64(981148)

### Geolocation Discovery Summary

| Metric | Finding |
|---|---|
| Grain | One row per zip/city/state/geolocation coordinate record |
| Candidate key | No single obvious key from raw data |
| Important columns | zip code prefix, latitude, longitude, city, state |
| Expected relationships | zip prefix can connect to customer and seller zip prefixes |
| Early quality notes | Zip prefixes repeat, so direct joins may multiply rows |
| Next checks | Build a deduplicated reference table by zip prefix before using it in joins |

## 4. Order Items Table

**Grain:** one row per item line inside an order.

**Candidate key:** likely `order_id` + `order_item_id`.

**Relationship notes:** joins to `orders`, `products`, and `sellers`. This is the main table for item revenue analysis.

In [41]:
discover_table(order_items, "order_items")

order_items dataset shape: (112650, 7)
Number of duplicate rows in order_items dataset: 0
Column data types:


,dtype
order_id,object
order_item_id,int64
product_id,object
seller_id,object
shipping_limit_date,object
price,float64
freight_value,float64


Missing values:


,missing_values
order_id,0
order_item_id,0
product_id,0
seller_id,0
shipping_limit_date,0
price,0
freight_value,0


Unique values per column:


,unique_values
order_id,98666
shipping_limit_date,93318
product_id,32951
freight_value,6999
price,5968
seller_id,3095
order_item_id,21


First 5 rows:


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


Description:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
order_id,112650,98666,8272b63d03f5f79c56e9e4120aec44ef,21,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_item_id,112650.0,NaN,NaN,NaN,1.197834,0.705124,1.0,1.0,1.0,1.0,21.0
product_id,112650,32951,aca2eb7d00ea1a7b8ebd4e68314663af,527,NaN,NaN,NaN,NaN,NaN,NaN,NaN
seller_id,112650,3095,6560211a19b47992c3666cc44a7e94c0,2033,NaN,NaN,NaN,NaN,NaN,NaN,NaN
shipping_limit_date,112650,93318,2017-07-21 18:25:23,21,NaN,NaN,NaN,NaN,NaN,NaN,NaN
price,112650.0,NaN,NaN,NaN,120.653739,183.633928,0.85,39.9,74.99,134.9,6735.0
freight_value,112650.0,NaN,NaN,NaN,19.99032,15.806405,0.0,13.08,16.26,21.15,409.68


In [42]:
order_items[["price", "freight_value"]].describe()

,price,freight_value
count,112650.000000,112650.000000
mean,120.653739,19.990320
std,183.633928,15.806405
min,0.850000,0.000000
25%,39.900000,13.080000
50%,74.990000,16.260000
75%,134.900000,21.150000
max,6735.000000,409.680000


In [43]:
order_items.groupby("order_id")["order_item_id"].count().describe()

count    98666.000000
mean         1.141731
std          0.538452
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         21.000000
Name: order_item_id, dtype: float64

### Order Items Discovery Summary

| Metric | Finding |
|---|---|
| Grain | One row per item line within an order |
| Candidate key | `order_id` + `order_item_id` |
| Important columns | `order_id`, `order_item_id`, `product_id`, `seller_id`, `price`, `freight_value` |
| Expected relationships | `order_id` -> `orders`; `product_id` -> `products`; `seller_id` -> `sellers` |
| Early quality notes | Counting `order_id` here counts item rows, not unique orders |
| Next checks | Use this table for revenue; use `COUNT(DISTINCT order_id)` for order count |

## 5. Order Payments Table

**Grain:** one row per payment sequence for an order.

**Candidate key:** likely `order_id` + `payment_sequential`.

**Relationship notes:** joins to `orders.order_id`. Some orders can have multiple payment rows.

In [44]:
discover_table(order_payments, "order_payments")

order_payments dataset shape: (103886, 5)
Number of duplicate rows in order_payments dataset: 0
Column data types:


,dtype
order_id,object
payment_sequential,int64
payment_type,object
payment_installments,int64
payment_value,float64


Missing values:


,missing_values
order_id,0
payment_sequential,0
payment_type,0
payment_installments,0
payment_value,0


Unique values per column:


,unique_values
order_id,99440
payment_value,29077
payment_sequential,29
payment_installments,24
payment_type,5


First 5 rows:


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


Description:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
order_id,103886,99440,fa65dad1b0e818e3ccc5cb0e39231352,29,NaN,NaN,NaN,NaN,NaN,NaN,NaN
payment_sequential,103886.0,NaN,NaN,NaN,1.092679,0.706584,1.0,1.0,1.0,1.0,29.0
payment_type,103886,5,credit_card,76795,NaN,NaN,NaN,NaN,NaN,NaN,NaN
payment_installments,103886.0,NaN,NaN,NaN,2.853349,2.687051,0.0,1.0,1.0,4.0,24.0
payment_value,103886.0,NaN,NaN,NaN,154.10038,217.494064,0.0,56.79,100.0,171.8375,13664.08


In [45]:
order_payments["payment_type"].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [46]:
order_payments[["payment_installments", "payment_value"]].describe()

,payment_installments,payment_value
count,103886.000000,103886.000000
mean,2.853349,154.100380
std,2.687051,217.494064
min,0.000000,0.000000
25%,1.000000,56.790000
50%,1.000000,100.000000
75%,4.000000,171.837500
max,24.000000,13664.080000


### Order Payments Discovery Summary

| Metric | Finding |
|---|---|
| Grain | One row per payment event/sequence for an order |
| Candidate key | `order_id` + `payment_sequential` |
| Important columns | `order_id`, `payment_type`, `payment_installments`, `payment_value` |
| Expected relationships | `order_id` -> `orders.order_id` |
| Early quality notes | Orders may have multiple payment rows |
| Next checks | Compare payment totals with item price plus freight; inspect payment types |

## 6. Order Reviews Table

**Grain:** one row per review record.

**Candidate key:** `review_id` may repeat in unusual cases, so verify uniqueness before treating it as a primary key.

**Relationship notes:** joins to `orders.order_id`. Useful for customer satisfaction and delivery analysis.

In [47]:
discover_table(order_reviews, "order_reviews")

order_reviews dataset shape: (99224, 7)
Number of duplicate rows in order_reviews dataset: 0
Column data types:


,dtype
review_id,object
order_id,object
review_score,int64
review_comment_title,object
review_comment_message,object
review_creation_date,object
review_answer_timestamp,object


Missing values:


,missing_values
review_id,0
order_id,0
review_score,0
review_comment_title,87656
review_comment_message,58247
review_creation_date,0
review_answer_timestamp,0


Unique values per column:


,unique_values
order_id,98673
review_id,98410
review_answer_timestamp,98248
review_comment_message,36159
review_comment_title,4527
review_creation_date,636
review_score,5


First 5 rows:


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


Description:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
review_id,99224,98410,7b606b0d57b078384f0b58eac1d41d78,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_id,99224,98673,c88b1d1b157a9999ce368f218a407141,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
review_score,99224.0,NaN,NaN,NaN,4.086421,1.347579,1.0,4.0,5.0,5.0,5.0
review_comment_title,11568,4527,Recomendo,423,NaN,NaN,NaN,NaN,NaN,NaN,NaN
review_comment_message,40977,36159,Muito bom,230,NaN,NaN,NaN,NaN,NaN,NaN,NaN
review_creation_date,99224,636,2017-12-19 00:00:00,463,NaN,NaN,NaN,NaN,NaN,NaN,NaN
review_answer_timestamp,99224,98248,2017-06-15 23:21:05,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [48]:
order_reviews["review_score"].value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [49]:
order_reviews["review_id"].duplicated().sum()

np.int64(814)

### Order Reviews Discovery Summary

| Metric | Finding |
|---|---|
| Grain | One row per review record |
| Candidate key | Check `review_id`; do not assume until duplicates are reviewed |
| Important columns | `review_id`, `order_id`, `review_score`, review timestamps, comment fields |
| Expected relationships | `order_id` -> `orders.order_id` |
| Early quality notes | Review comment fields are often missing, which can be normal |
| Next checks | Compare review score with delivery duration and late delivery status |

## 7. Products Table

**Grain:** one row per product.

**Candidate key:** `product_id`.

**Relationship notes:** joins to `order_items.product_id`; category name can join to `categories.product_category_name`.

In [50]:
discover_table(products, "products")

products dataset shape: (32951, 9)
Number of duplicate rows in products dataset: 0
Column data types:


,dtype
product_id,object
product_category_name,object
product_name_lenght,float64
product_description_lenght,float64
product_photos_qty,float64
product_weight_g,float64
product_length_cm,float64
product_height_cm,float64
product_width_cm,float64


Missing values:


,missing_values
product_id,0
product_category_name,610
product_name_lenght,610
product_description_lenght,610
product_photos_qty,610
product_weight_g,2
product_length_cm,2
product_height_cm,2
product_width_cm,2


Unique values per column:


,unique_values
product_id,32951
product_description_lenght,2960
product_weight_g,2204
product_height_cm,102
product_length_cm,99
product_width_cm,95
product_category_name,73
product_name_lenght,66
product_photos_qty,19


First 5 rows:


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


Description:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
product_id,32951,32951,1e9e8ef04dbcff4541ed26657ea517e5,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product_category_name,32341,73,cama_mesa_banho,3029,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product_name_lenght,32341.0,NaN,NaN,NaN,48.476949,10.245741,5.0,42.0,51.0,57.0,76.0
product_description_lenght,32341.0,NaN,NaN,NaN,771.495285,635.115225,4.0,339.0,595.0,972.0,3992.0
product_photos_qty,32341.0,NaN,NaN,NaN,2.188986,1.736766,1.0,1.0,1.0,3.0,20.0
product_weight_g,32949.0,NaN,NaN,NaN,2276.472488,4282.038731,0.0,300.0,700.0,1900.0,40425.0
product_length_cm,32949.0,NaN,NaN,NaN,30.815078,16.914458,7.0,18.0,25.0,38.0,105.0
product_height_cm,32949.0,NaN,NaN,NaN,16.937661,13.637554,2.0,8.0,13.0,21.0,105.0
product_width_cm,32949.0,NaN,NaN,NaN,23.196728,12.079047,6.0,15.0,20.0,30.0,118.0


In [51]:
products["product_category_name"].value_counts().head(10)

product_category_name
cama_mesa_banho           3029
esporte_lazer             2867
moveis_decoracao          2657
beleza_saude              2444
utilidades_domesticas     2335
automotivo                1900
informatica_acessorios    1639
brinquedos                1411
relogios_presentes        1329
telefonia                 1134
Name: count, dtype: int64

In [52]:
products.isna().sum().sort_values(ascending=False)

product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
product_id                      0
dtype: int64

### Products Discovery Summary

| Metric | Finding |
|---|---|
| Grain | One row per product |
| Candidate key | `product_id` |
| Important columns | category, name/description/photo counts, weight and dimensions |
| Expected relationships | `product_id` -> `order_items.product_id`; category -> category translation table |
| Early quality notes | Product attributes and category can contain missing values |
| Next checks | Translate categories and inspect product/category revenue after joining to order items |

## 8. Sellers Table

**Grain:** one row per seller.

**Candidate key:** `seller_id`.

**Relationship notes:** joins to `order_items.seller_id`. Seller zip/state/city support geography analysis.

In [53]:
discover_table(sellers, "sellers")

sellers dataset shape: (3095, 4)
Number of duplicate rows in sellers dataset: 0
Column data types:


,dtype
seller_id,object
seller_zip_code_prefix,int64
seller_city,object
seller_state,object


Missing values:


,missing_values
seller_id,0
seller_zip_code_prefix,0
seller_city,0
seller_state,0


Unique values per column:


,unique_values
seller_id,3095
seller_zip_code_prefix,2246
seller_city,611
seller_state,23


First 5 rows:


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


Description:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
seller_id,3095,3095,3442f8959a84dea7ee197c632cb2df15,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
seller_zip_code_prefix,3095.0,NaN,NaN,NaN,32291.059451,32713.45383,1001.0,7093.5,14940.0,64552.5,99730.0
seller_city,3095,611,sao paulo,694,NaN,NaN,NaN,NaN,NaN,NaN,NaN
seller_state,3095,23,SP,1849,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [54]:
sellers["seller_state"].value_counts().head(10)

seller_state
SP    1849
PR     349
MG     244
SC     190
RJ     171
RS     129
GO      40
DF      30
ES      23
BA      19
Name: count, dtype: int64

### Sellers Discovery Summary

| Metric | Finding |
|---|---|
| Grain | One row per seller |
| Candidate key | `seller_id` |
| Important columns | seller ID, zip code prefix, city, state |
| Expected relationships | `seller_id` -> `order_items.seller_id` |
| Early quality notes | Seller geography can be compared with customer geography later |
| Next checks | Analyze seller revenue, order volume, and delivery/review performance |

## 9. Category Translation Table

**Grain:** one row per product category translation.

**Candidate key:** `product_category_name`.

**Relationship notes:** joins to `products.product_category_name` to create readable English category labels.

In [55]:
discover_table(categories, "categories")

categories dataset shape: (71, 2)
Number of duplicate rows in categories dataset: 0
Column data types:


,dtype
product_category_name,object
product_category_name_english,object


Missing values:


,missing_values
product_category_name,0
product_category_name_english,0


Unique values per column:


,unique_values
product_category_name,71
product_category_name_english,71


First 5 rows:


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


Description:


,count,unique,top,freq
product_category_name,71,71,beleza_saude,1
product_category_name_english,71,71,health_beauty,1


In [56]:
categories.head(10)

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor
5,esporte_lazer,sports_leisure
6,perfumaria,perfumery
7,utilidades_domesticas,housewares
8,telefonia,telephony
9,relogios_presentes,watches_gifts


### Category Translation Discovery Summary

| Metric | Finding |
|---|---|
| Grain | One row per product category translation |
| Candidate key | `product_category_name` |
| Important columns | Portuguese category name and English category name |
| Expected relationships | `product_category_name` -> `products.product_category_name` |
| Early quality notes | This is a reference table, not a fact table |
| Next checks | Find product categories without an English translation |

## Relationship Checks

These checks help confirm whether the expected joins are valid before doing EDA or building SQL tables.

In [57]:
relationship_checks = {
    "orders missing customer": (~orders["customer_id"].isin(customers["customer_id"])).sum(),
    "order_items missing order": (~order_items["order_id"].isin(orders["order_id"])).sum(),
    "payments missing order": (~order_payments["order_id"].isin(orders["order_id"])).sum(),
    "reviews missing order": (~order_reviews["order_id"].isin(orders["order_id"])).sum(),
    "order_items missing product": (~order_items["product_id"].isin(products["product_id"])).sum(),
    "order_items missing seller": (~order_items["seller_id"].isin(sellers["seller_id"])).sum(),
    "products missing category translation": (
        products["product_category_name"].notna()
        & ~products["product_category_name"].isin(categories["product_category_name"])
    ).sum(),
}

pd.Series(relationship_checks, name="missing_fk_count").to_frame()

,missing_fk_count
orders missing customer,0
order_items missing order,0
payments missing order,0
reviews missing order,0
order_items missing product,0
order_items missing seller,0
products missing category translation,13


## Discovery Notes to Carry Forward

- `orders` is order-level, while `order_items` is item-level. Be careful not to inflate order counts after joining.
- Timestamp columns load as text and should be converted to datetime before EDA.
- `geolocation` has repeated zip prefixes, so it needs deduplication or aggregation before it becomes a clean reference table.
- `customer_unique_id` is the field to use for repeat-customer analysis, not `customer_id`.
- The next notebook, `02_eda.ipynb`, should start with trusted joins and business metrics: revenue, orders, AOV, categories, sellers, geography, delivery time, and review score.